In [ ]:
import pandas as pd
import geopandas as gpd
import contextily as ctx
from matplotlib import pyplot as plt

city = "Edinburgh"

cell_features = pd.read_parquet(f"data/{city}/cell_features.parquet")
cell_features.index.set_names("cell_id", inplace=True)

hex_grid = gpd.read_parquet(f"data/{city}/hex_grid.geoparquet")

stations = gpd.read_parquet(f"data/{city}/trips/stations.geoparquet")

In [ ]:
columns = cell_features.columns
columns = cell_features.isna().mean().sort_values()[:20].index

ncols = 4
nrows = len(columns) // ncols + 1
fig, axes = plt.subplots(nrows, ncols, figsize=(16,16))
axes=axes.flatten()

for ax, col in zip(axes, columns):
    hex_grid.plot(alpha=.5, column=cell_features[col].fillna(0), edgecolor='w', ax=ax, cmap='Reds')
    ctx.add_basemap(ax=ax, crs=hex_grid.crs, attribution='')
    ax.set_title(col)
plt.tight_layout()


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, Normalizer, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.neighbors import KNeighborsClassifier
from sklearn import metrics
import numpy as np
from matplotlib import pyplot as plt

classifer_pipeline = Pipeline([
    ("std", Normalizer('l1')),
    # ("std", Normalizer('l2')),
    # ("std", StandardScaler()),
    # ("pca", PCA(.5)),
    # ("std", MinMaxScaler()),
    ("clf", KMeans())
])

data = cell_features.fillna(0).values

cluster_range = range(2, 20)
silhouettes = []
silhouettes_std = []

calinski_harabasz = []
calinski_harabasz_std = []

for n_clusters in cluster_range:
    sil = []
    ch = []
    for i in range(50):
        classifer_pipeline.set_params(clf__n_clusters = n_clusters)
        labels = classifer_pipeline.fit_predict(data)
        sil.append(metrics.silhouette_score(data, labels, metric='euclidean'))
        ch.append(metrics.calinski_harabasz_score(data, labels))
    
    silhouettes.append(np.mean(sil))
    silhouettes_std.append(np.std(sil))
    calinski_harabasz.append(np.mean(ch))
    calinski_harabasz_std.append(np.std(ch))

plt.figure(figsize=(12,4))
plt.errorbar(cluster_range, silhouettes, marker='o', yerr=silhouettes_std)
plt.grid()
plt.show()

plt.figure(figsize=(12,4))
plt.errorbar(cluster_range, calinski_harabasz, marker='o', yerr=calinski_harabasz_std)
plt.grid()
plt.show()

In [ ]:
n_clusters = 5
classifer_pipeline.set_params(clf__n_clusters = n_clusters)
labels = classifer_pipeline.fit_predict(data)
labels = pd.Series(labels, index=cell_features.index)

ax = hex_grid.plot(column=labels, alpha=.5, figsize=(12,6), cmap='tab10', legend = True)
ctx.add_basemap(ax=ax, crs=hex_grid.crs)

In [ ]:
kmeans = classifer_pipeline['clf']
centroids = kmeans.cluster_centers_

centroids = pd.DataFrame(centroids, columns = cell_features.columns).T

for c in range(n_clusters):
    print(c, centroids[c].sort_values(ascending=False).head(10).round(2).to_dict())

In [ ]:
labels.to_pickle(f"data/{city}/cell_clusters.pkl")
centroids.to_pickle(f"data/{city}/cluster_centers.pkl")